# Exa.ai Website Monitoring & Navigation - Testing Notebook

This notebook demonstrates various approaches for monitoring specific websites and extracting content from multiple pages using Exa.ai.

## Table of Contents
1. Setup & Configuration
2. Basic Website Monitoring
3. Subpage Crawling & Navigation
4. Batch URL Processing
5. Sitemap Navigation
6. Advanced Approaches
7. Practical Use Cases
8. Comparison & Best Practices

## 1. Setup & Configuration

In [ ]:
# Install required packages (uncomment if needed)
!pip install exa_py requests xmltodict python-dotenv

In [5]:
import os
import json
from datetime import datetime, timedelta
from typing import List, Dict, Any
import time

# Exa imports
from exa_py import Exa

# For sitemap parsing
import requests
import xml.etree.ElementTree as ET

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

print("✅ Imports successful!")

✅ Imports successful!


In [6]:
# Configuration
EXA_API_KEY = os.getenv("EXA_API_KEY", "your-api-key-here")

# if EXA_API_KEY == "your-api-key-here":
#     print("⚠️  Please set your EXA_API_KEY in .env file or replace above")
# else:
#     print("✅ API key loaded")

# Initialize Exa client
exa = Exa(api_key=EXA_API_KEY)
print("✅ Exa client initialized")

✅ Exa client initialized


In [15]:
# Helper function to display results nicely
def display_results(results, max_items=3):
    """
    Display Exa search/content results in a readable format.
    """
    if hasattr(results, 'results'):
        items = results.results
    elif isinstance(results, list):
        items = results
    else:
        items = [results]
    
    print(f"\n📊 Found {len(items)} results\n")
    print("=" * 80)
    
    for i, item in enumerate(items[:max_items], 1):
        print(f"\n{i}. {getattr(item, 'title', 'No title')}")
        print(f"   URL: {getattr(item, 'url', 'N/A')}")
        print(f"   Published: {getattr(item, 'published_date', 'N/A')}")
        
        # Display text content if available
        if hasattr(item, 'text') and item.text:
            # text_preview = item.text[:200] + "..." if len(item.text) > 200 else item.text
            # print(f"   Content: {text_preview}")
            print("Full Text" , item.text)
        
        # Display highlights if available
        if hasattr(item, 'highlights') and item.highlights:
            print(f"   Highlights: {item.highlights[:1]}")
        
        print("-" * 80)
    
    if len(items) > max_items:
        print(f"\n... and {len(items) - max_items} more results\n")

print("✅ Helper functions loaded")

✅ Helper functions loaded


## 2. Basic Website Monitoring

Monitor specific websites for new content using domain and date filtering.

### 2.1 Domain Filtering - Include Specific Domains

In [ ]:
# Example: Monitor EU Commission website only
print("🔍 Searching EU Commission press releases...\n")

start_time = time.time()

results = exa.search_and_contents(
    "digital services act enforcement",
    include_domains=["ec.europa.eu"],  # Only search this domain
    num_results=5,
    text=True,
    use_autoprompt=False
)

execution_time = time.time() - start_time
print(f"⏱️  Execution time: {execution_time:.2f}s")

display_results(results)

### 2.2 Date Filtering - Get Recent Content Only

In [ ]:
# Example: Get content published in the last 7 days
print("📅 Searching for content from last 7 days...\n")

# Calculate date range
end_date = datetime.now()
start_date = end_date - timedelta(days=7)
start_published_date = start_date.strftime("%Y-%m-%dT%H:%M:%S.000Z")

print(f"Date range: {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}\n")

results = exa.search_and_contents(
    "AI regulation policy updates",
    include_domains=["ec.europa.eu", "edpb.europa.eu"],
    start_published_date=start_published_date,
    num_results=5,
    text=True
)

display_results(results)

### 2.3 Live Crawling - Force Fresh Data

In [ ]:
# Example: Always get the freshest data from a specific page
print("🔴 Live crawling for fresh data...\n")

# Specific URL you want to monitor
target_url = "https://ec.europa.eu/commission/presscorner/home/en"

results = exa.get_contents(
    [target_url],
    text=True,
    livecrawl="always"  # Options: "always", "preferred", "fallback", "never"
)

display_results(results)

### 2.4 Crawl Date Filtering - Track When Exa Discovered Content

In [ ]:
# Example: Find content that Exa crawled in the last 3 days
# This is useful for finding "newly discovered" content
print("🕷️  Finding newly crawled content...\n")

crawl_start_date = (datetime.now() - timedelta(days=3)).strftime("%Y-%m-%dT%H:%M:%S.000Z")

results = exa.search_and_contents(
    "site:ec.europa.eu digital policy",
    start_crawl_date=crawl_start_date,
    num_results=5,
    text=True
)

display_results(results)

## 3. Subpage Crawling & Navigation

Automatically discover and crawl related pages from a starting URL.

### 3.1 Basic Subpage Crawling

In [16]:
# Example: Start from a main page and crawl related subpages
print("🌐 Crawling main page + subpages...\n")

# Starting URL
main_url = "https://www.bundestag.de/abgeordnete/biografien/A/abdi_sanae-1043330"

results = exa.get_contents(
    [main_url],
    subpages=5,  # Crawl up to 5 additional pages
    text=True
)

print(f"📄 Crawled {len(results.results)} pages total (main page + subpages)\n")
display_results(results, max_items=6)

🌐 Crawling main page + subpages...

📄 Crawled 1 pages total (main page + subpages)


📊 Found 1 results


1. Deutscher Bundestag - Sanae Abdi
   URL: https://bundestag.de/abgeordnete/biografien/A/abdi_sanae-1043330
   Published: 2025-04-01T00:00:00.000Z
Full Text alles öffnenalles schließen

Biografie

Geboren am 7. Juli 1986 in Tetouan, Marokko; ledig.

2005 Abitur an der Städtischen Adolf-Reichwein-Gesamtschule in Lüdenscheid. Anschließendes Jurastudium in Marburg, Bonn und Köln.

Seit 2008 Mitglied der Sozialdemokratischen Partei Deutschlands (SPD).

2013 bis 2018 wissenschaftliche Mitarbeiterin in Kölner Wirtschaftskanzlei und freiberufliche Tätigkeit für Kölner Strafrechtskanzlei.; seit 2018 Projektmanagerin mit Schwerpunkt Controlling bei der Deutschen Gesellschaft für internationale Zusammenarbeit (GIZ).

Seit 2021 Mitglied des Deutschen Bundestages; 2021 bis 2025 Obfrau des Ausschuss für wirtschaftliche Zusammenarbeit und Entwicklung und Entwicklungspolitische Sprecherin der SPD

In [14]:
results

SearchResponse(results=[Result(url='https://bundestag.de/abgeordnete/biografien/A/abdi_sanae-1043330', id='https://www.bundestag.de/abgeordnete/biografien/A/abdi_sanae-1043330', title='Deutscher Bundestag - Sanae Abdi', score=None, published_date='2025-04-01T00:00:00.000Z', author=None, image=None, favicon=None, subpages=[], extras=None, text='alles öffnenalles schließen\n\nBiografie\n\nGeboren am 7. Juli 1986 in Tetouan, Marokko; ledig.\n\n2005 Abitur an der Städtischen Adolf-Reichwein-Gesamtschule in Lüdenscheid. Anschließendes Jurastudium in Marburg, Bonn und Köln.\n\nSeit 2008 Mitglied der Sozialdemokratischen Partei Deutschlands (SPD).\n\n2013 bis 2018 wissenschaftliche Mitarbeiterin in Kölner Wirtschaftskanzlei und freiberufliche Tätigkeit für Kölner Strafrechtskanzlei.; seit 2018 Projektmanagerin mit Schwerpunkt Controlling bei der Deutschen Gesellschaft für internationale Zusammenarbeit (GIZ).\n\nSeit 2021 Mitglied des Deutschen Bundestages; 2021 bis 2025 Obfrau des Ausschuss f

### 3.2 Targeted Subpage Crawling with Keywords

In [ ]:
# Example: Crawl subpages but prioritize pages containing specific keywords
print("🎯 Targeted subpage crawling with keywords...\n")

results = exa.get_contents(
    ["https://ec.europa.eu/info/index_en"],
    subpages=10,
    subpage_target=["digital", "regulation", "policy"],  # Prioritize pages with these terms
    text=True
)

print(f"📄 Found {len(results.results)} pages related to target keywords\n")
display_results(results, max_items=5)

### 3.3 Crawl Documentation/News Sections

In [ ]:
# Example: Navigate through a documentation or news section
print("📚 Crawling documentation section...\n")

# Start from docs page and find related documentation
results = exa.search_and_contents(
    "site:ec.europa.eu digital services act documentation",
    include_domains=["ec.europa.eu"],
    num_results=3,
    subpages=5,
    subpage_target=["guide", "documentation", "implementation"],
    text=True
)

display_results(results, max_items=8)

## 4. Batch URL Processing

Process multiple specific URLs in a single request.

### 4.1 Crawl Multiple Known URLs

In [ ]:
# Example: You have a list of specific URLs to monitor
print("📋 Batch processing multiple URLs...\n")

urls_to_monitor = [
    "https://ec.europa.eu/commission/presscorner/home/en",
    "https://edpb.europa.eu/news/news_en",
    "https://eur-lex.europa.eu/homepage.html"
]

print(f"Processing {len(urls_to_monitor)} URLs...\n")

results = exa.get_contents(
    urls_to_monitor,
    text=True,
    livecrawl="preferred"  # Try fresh crawl, fall back to cache
)

display_results(results, max_items=10)

### 4.2 Different Content Extraction Options

In [ ]:
# Example: Extract highlights and summaries instead of full text
print("📝 Extracting highlights and summaries...\n")

urls = [
    "https://ec.europa.eu/commission/presscorner/home/en"
]

results = exa.get_contents(
    urls,
    highlights={
        "num_sentences": 3,
        "query": "digital policy regulation"
    },
    summary={
        "query": "What are the main policy updates?"
    }
)

for i, result in enumerate(results.results, 1):
    print(f"\n{i}. {result.title}")
    print(f"   URL: {result.url}")
    if hasattr(result, 'highlights') and result.highlights:
        print(f"\n   🔦 Highlights:")
        for highlight in result.highlights:
            print(f"      • {highlight}")
    if hasattr(result, 'summary') and result.summary:
        print(f"\n   📋 Summary: {result.summary}")
    print("-" * 80)

## 5. Sitemap Navigation

Parse a sitemap and crawl specific pages.

### 5.1 Parse Sitemap XML

In [ ]:
def parse_sitemap(sitemap_url: str) -> List[Dict[str, Any]]:
    """
    Parse a sitemap.xml file and extract URLs with metadata.
    
    Returns list of dicts with: url, lastmod, changefreq, priority
    """
    try:
        print(f"📥 Fetching sitemap from {sitemap_url}...")
        response = requests.get(sitemap_url, timeout=30)
        response.raise_for_status()
        
        print("✅ Sitemap downloaded, parsing...")
        root = ET.fromstring(response.content)
        
        # Handle sitemap namespace
        ns = {'sm': 'http://www.sitemaps.org/schemas/sitemap/0.9'}
        
        urls = []
        for url_elem in root.findall('.//sm:url', ns):
            loc = url_elem.find('sm:loc', ns)
            lastmod = url_elem.find('sm:lastmod', ns)
            changefreq = url_elem.find('sm:changefreq', ns)
            priority = url_elem.find('sm:priority', ns)
            
            urls.append({
                'url': loc.text if loc is not None else None,
                'lastmod': lastmod.text if lastmod is not None else None,
                'changefreq': changefreq.text if changefreq is not None else None,
                'priority': priority.text if priority is not None else None
            })
        
        print(f"✅ Parsed {len(urls)} URLs from sitemap\n")
        return urls
        
    except Exception as e:
        print(f"❌ Error parsing sitemap: {e}")
        return []

# Example: Parse a sitemap (replace with actual sitemap URL)
# Note: Many government sites don't have public sitemaps, so this is a generic example
sitemap_url = "https://www.example.com/sitemap.xml"  # Replace with real sitemap

print("\n⚠️  Note: Replace sitemap_url with an actual sitemap to test\n")
print(f"Example sitemap URL: {sitemap_url}\n")

# Uncomment to test with real sitemap:
# sitemap_urls = parse_sitemap(sitemap_url)
# print(f"Sample URLs from sitemap:")
# for url_data in sitemap_urls[:5]:
#     print(f"  • {url_data['url']} (last modified: {url_data['lastmod']})")

### 5.2 Filter and Crawl URLs from Sitemap

In [ ]:
def filter_urls_by_pattern(urls: List[Dict], patterns: List[str]) -> List[str]:
    """
    Filter URLs that match any of the given patterns.
    """
    filtered = []
    for url_data in urls:
        url = url_data.get('url', '')
        if any(pattern in url for pattern in patterns):
            filtered.append(url)
    return filtered

def filter_urls_by_date(urls: List[Dict], days_back: int = 7) -> List[str]:
    """
    Filter URLs modified in the last N days.
    """
    cutoff_date = datetime.now() - timedelta(days=days_back)
    filtered = []
    
    for url_data in urls:
        lastmod = url_data.get('lastmod')
        if lastmod:
            try:
                # Parse date (handles various ISO formats)
                mod_date = datetime.fromisoformat(lastmod.replace('Z', '+00:00'))
                if mod_date >= cutoff_date:
                    filtered.append(url_data.get('url'))
            except (ValueError, AttributeError):
                continue
    
    return filtered

# Example workflow
print("🔍 Sitemap-based crawling workflow:\n")
print("1. Parse sitemap.xml")
print("2. Filter URLs by pattern (e.g., /news/, /press-release/)")
print("3. Filter URLs by date (last 7 days)")
print("4. Batch crawl filtered URLs with Exa\n")

# Simulated example (replace with real sitemap)
example_sitemap_data = [
    {'url': 'https://example.com/news/article1', 'lastmod': '2024-11-03T10:00:00Z'},
    {'url': 'https://example.com/news/article2', 'lastmod': '2024-11-01T10:00:00Z'},
    {'url': 'https://example.com/about', 'lastmod': '2024-10-15T10:00:00Z'},
    {'url': 'https://example.com/press-release/item1', 'lastmod': '2024-11-04T10:00:00Z'},
]

# Filter by pattern
news_urls = filter_urls_by_pattern(example_sitemap_data, ['/news/', '/press-release/'])
print(f"URLs matching pattern: {len(news_urls)}")
for url in news_urls:
    print(f"  • {url}")

# Filter by date
recent_urls = filter_urls_by_date(example_sitemap_data, days_back=7)
print(f"\nURLs from last 7 days: {len(recent_urls)}")
for url in recent_urls:
    print(f"  • {url}")

# Crawl with Exa (uncomment to test with real URLs)
# if recent_urls:
#     print(f"\n📥 Crawling {len(recent_urls)} filtered URLs...")
#     results = exa.get_contents(recent_urls, text=True)
#     display_results(results)

## 6. Advanced Approaches

Combining multiple techniques for comprehensive monitoring.

### 6.1 Search + Domain + Date + Subpages

In [ ]:
# Example: Comprehensive monitoring approach
print("🎯 Advanced: Search + Domain + Date + Subpage Crawling\n")

start_date = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%dT%H:%M:%S.000Z")

results = exa.search_and_contents(
    "digital services act enforcement actions",
    
    # Domain filtering
    include_domains=["ec.europa.eu", "edpb.europa.eu"],
    
    # Date filtering
    start_published_date=start_date,
    
    # Search parameters
    num_results=3,
    category="news",
    
    # Subpage crawling
    subpages=5,
    subpage_target=["enforcement", "compliance", "violation"],
    
    # Content extraction
    text=True,
    highlights={"num_sentences": 2, "query": "enforcement"},
    
    # Force fresh data
    livecrawl="preferred"
)

print(f"📊 Total pages retrieved: {len(results.results)}\n")
display_results(results, max_items=10)

### 6.2 Multi-Domain Monitoring

In [ ]:
# Example: Monitor multiple government/regulatory websites
print("🌐 Multi-domain monitoring setup\n")

target_domains = [
    "ec.europa.eu",           # EU Commission
    "edpb.europa.eu",         # European Data Protection Board
    "eur-lex.europa.eu",      # EU Legal documents
]

print("Target domains:")
for domain in target_domains:
    print(f"  • {domain}")

print("\n🔍 Searching across all domains...\n")

results = exa.search_and_contents(
    "AI regulation artificial intelligence policy",
    include_domains=target_domains,
    start_published_date=(datetime.now() - timedelta(days=14)).strftime("%Y-%m-%dT%H:%M:%S.000Z"),
    num_results=10,
    text=True
)

# Group results by domain
from urllib.parse import urlparse
by_domain = {}
for result in results.results:
    domain = urlparse(result.url).netloc
    if domain not in by_domain:
        by_domain[domain] = []
    by_domain[domain].append(result)

print("\n📊 Results by domain:\n")
for domain, items in by_domain.items():
    print(f"  {domain}: {len(items)} results")

display_results(results, max_items=10)

### 6.3 Periodic Monitoring Setup

In [ ]:
# Example: Simulated periodic monitoring
def monitor_website(
    query: str,
    domains: List[str],
    hours_back: int = 24
) -> Dict[str, Any]:
    """
    Monitor specific domains for new content.
    Returns summary of findings.
    """
    start_date = (datetime.now() - timedelta(hours=hours_back)).strftime("%Y-%m-%dT%H:%M:%S.000Z")
    
    print(f"🔍 Monitoring query: '{query}'")
    print(f"📅 Time range: Last {hours_back} hours")
    print(f"🌐 Domains: {', '.join(domains)}\n")
    
    results = exa.search_and_contents(
        query,
        include_domains=domains,
        start_crawl_date=start_date,  # Use crawl date for new discoveries
        num_results=20,
        text=True
    )
    
    return {
        "timestamp": datetime.now().isoformat(),
        "query": query,
        "domains": domains,
        "hours_back": hours_back,
        "results_found": len(results.results),
        "results": results
    }

# Example monitoring job
print("⏰ Periodic Monitoring Example\n")
print("This would be run on a schedule (hourly/daily)\n")

monitoring_result = monitor_website(
    query="digital policy updates regulations",
    domains=["ec.europa.eu"],
    hours_back=72  # Last 3 days
)

print(f"\n📊 Monitoring Summary:")
print(f"  Timestamp: {monitoring_result['timestamp']}")
print(f"  Results found: {monitoring_result['results_found']}\n")

if monitoring_result['results_found'] > 0:
    print("🔔 New content detected!\n")
    display_results(monitoring_result['results'], max_items=5)
else:
    print("✅ No new content found in this time period")

## 7. Practical Use Cases

Real-world scenarios for website monitoring.

### 7.1 Monitor Government Press Releases

In [ ]:
# Use Case: Track EU Commission press releases
print("📰 Use Case: Monitor EU Commission Press Releases\n")

results = exa.search_and_contents(
    "site:ec.europa.eu/commission/presscorner",
    include_domains=["ec.europa.eu"],
    start_published_date=(datetime.now() - timedelta(days=7)).strftime("%Y-%m-%dT%H:%M:%S.000Z"),
    num_results=5,
    category="news",
    text=True,
    highlights={"num_sentences": 3}
)

print(f"Found {len(results.results)} press releases from last 7 days\n")
display_results(results)

### 7.2 Track Regulatory Documents

In [ ]:
# Use Case: Monitor new regulations and legal texts
print("⚖️  Use Case: Track New Regulatory Documents\n")

results = exa.search_and_contents(
    "regulation directive decision",
    include_domains=["eur-lex.europa.eu"],
    start_published_date=(datetime.now() - timedelta(days=30)).strftime("%Y-%m-%dT%H:%M:%S.000Z"),
    num_results=5,
    text=True
)

print(f"Found {len(results.results)} regulatory documents from last 30 days\n")
display_results(results)

### 7.3 Monitor Specific Policy Areas

In [ ]:
# Use Case: Track specific policy topics across multiple sources
print("🎯 Use Case: Monitor AI Policy Developments\n")

policy_keywords = [
    "artificial intelligence AI act",
    "algorithmic transparency",
    "high-risk AI systems"
]

target_sites = [
    "ec.europa.eu",
    "edpb.europa.eu",
    "eur-lex.europa.eu"
]

all_results = []
for keyword in policy_keywords:
    print(f"\n🔍 Searching: '{keyword}'")
    results = exa.search_and_contents(
        keyword,
        include_domains=target_sites,
        start_published_date=(datetime.now() - timedelta(days=14)).strftime("%Y-%m-%dT%H:%M:%S.000Z"),
        num_results=3,
        text=True
    )
    all_results.extend(results.results)
    print(f"  Found: {len(results.results)} documents")

print(f"\n📊 Total documents found: {len(all_results)}\n")
print("Sample results:\n")
for i, result in enumerate(all_results[:5], 1):
    print(f"{i}. {result.title}")
    print(f"   {result.url}\n")

## 8. Comparison & Best Practices

When to use each approach and optimization tips.

### 8.1 Approach Comparison

In [ ]:
# Comparison of different approaches
print("""🔍 APPROACH COMPARISON

1️⃣  DOMAIN + DATE FILTERING (search_and_contents)
   ✅ Best for: Discovering new content on known domains
   ✅ Use when: You want semantic search + domain restrictions
   ⏱️  Speed: Fast (uses Exa's index)
   💰 Cost: Per search + per result

2️⃣  SUBPAGE CRAWLING (get_contents with subpages)
   ✅ Best for: Exploring unknown website structure
   ✅ Use when: Starting from a main page, need related content
   ⏱️  Speed: Medium (follows links intelligently)
   💰 Cost: Per URL + per subpage

3️⃣  BATCH URL PROCESSING (get_contents with URL list)
   ✅ Best for: Known URLs (from sitemap or previous crawls)
   ✅ Use when: You have exact pages to monitor
   ⏱️  Speed: Fast (direct URL access)
   💰 Cost: Per URL crawled

4️⃣  SITEMAP PARSING + BATCH CRAWLING
   ✅ Best for: Comprehensive site monitoring
   ✅ Use when: Site has sitemap, need systematic coverage
   ⏱️  Speed: Fast (after sitemap parsing)
   💰 Cost: Sitemap fetch (free) + per URL

5️⃣  LIVE CRAWLING (livecrawl="always")
   ✅ Best for: Critical updates, real-time monitoring
   ✅ Use when: Need absolutely fresh data
   ⏱️  Speed: Slower (live page fetch)
   💰 Cost: Higher (live crawl cost)

6️⃣  WEBSETS API (continuous monitoring)
   ✅ Best for: Production monitoring with webhooks
   ✅ Use when: Need automated daily/hourly updates
   ⏱️  Speed: Background process
   💰 Cost: Subscription-based
""")

### 8.2 Best Practices

In [ ]:
print("""💡 BEST PRACTICES

🎯 CHOOSING THE RIGHT APPROACH:
   • Start with domain + date filtering for discovery
   • Use subpage crawling for deep dives
   • Switch to batch URL processing once you know the pages
   • Use live crawling only when cache is not acceptable

⚡ PERFORMANCE OPTIMIZATION:
   • Batch URLs when possible (up to limits)
   • Use cached results for historical data
   • Filter URLs before crawling (save API calls)
   • Use highlights/summaries instead of full text when possible

💰 COST OPTIMIZATION:
   • Prefer cached results over live crawling
   • Use semantic search to reduce result count
   • Filter by date to limit results
   • Use highlights instead of full text for previews

🔄 MONITORING STRATEGY:
   • Use startCrawlDate for new discoveries
   • Use startPublishedDate for content updates
   • Combine both for comprehensive monitoring
   • Track processed URLs to avoid duplicates

📊 DATA MANAGEMENT:
   • Store crawled URLs to track coverage
   • Track lastmod dates from sitemaps
   • Deduplicate results by URL
   • Save content hashes to detect changes

🛡️  ERROR HANDLING:
   • Handle rate limits gracefully
   • Retry failed requests with backoff
   • Validate URLs before crawling
   • Log errors for debugging
""")

### 8.3 Sample Monitoring Workflow

In [ ]:
print("""🔄 RECOMMENDED MONITORING WORKFLOW

STEP 1: Initial Discovery (Run once or weekly)
   → Use domain + date filtering to find relevant pages
   → Store discovered URLs
   → Build initial URL database

STEP 2: Ongoing Monitoring (Run daily)
   → Use startCrawlDate to find newly discovered content
   → Check known URLs for updates (batch processing)
   → Add new URLs to database

STEP 3: Deep Dive (Run on demand)
   → Use subpage crawling for detailed exploration
   → Follow links from important pages
   → Extract full text content

STEP 4: Real-time Updates (Critical content only)
   → Use live crawling for time-sensitive pages
   → Set up Websets monitors with webhooks
   → Immediate alerts for important updates

IMPLEMENTATION TIPS:
   ✅ Start simple: domain + date filtering
   ✅ Build URL database incrementally
   ✅ Add complexity as needed
   ✅ Monitor costs and optimize
   ✅ Track what works for your use case
""")

## 9. Next Steps

Ready to implement in production? Here's what to do:

In [ ]:
print("""🚀 READY FOR PRODUCTION?

TO INTEGRATE INTO YOUR ETL PIPELINE:

1. Add to ExaDirectCollector:
   • Add include_domains parameter
   • Add exclude_domains parameter
   • Add livecrawl option
   • Add start_crawl_date support

2. Create WebsiteMonitorCollector:
   • Dedicated collector for specific sites
   • Support for URL lists
   • Subpage crawling support
   • Sitemap parsing integration

3. Add Airflow DAG:
   • Schedule: hourly/daily based on site
   • Track processed URLs
   • Store in data/input/websites/
   • Integration with Flow 1

4. Configuration:
   • Add monitored_websites to client.yaml
   • Specify update frequencies
   • Set crawling depths
   • Define content filters

EXAMPLE CONFIGURATION (data/context/client.yaml):

monitored_websites:
  high_frequency:  # Hourly
    - domain: "ec.europa.eu/commission/presscorner"
      name: "EU Commission Press"
      subpages: 5
      keywords: ["digital", "policy", "regulation"]
  
  daily:
    - domain: "edpb.europa.eu"
      name: "EDPB News"
      subpages: 10
      sitemap: "https://edpb.europa.eu/sitemap.xml"

Would you like me to implement any of these integrations?
""")

---

## Summary

This notebook demonstrated:
- ✅ Domain filtering for specific websites
- ✅ Date filtering for recent content
- ✅ Subpage crawling with keyword targeting
- ✅ Batch URL processing
- ✅ Sitemap parsing and navigation
- ✅ Live crawling for fresh data
- ✅ Combined approaches for comprehensive monitoring

**Ready to integrate? Let me know which approach works best for your needs!**